# DQN Boxing â€” wagner's experiments (Colab)
Run cells top to bottom. Uses a free GPU (Runtime > Change runtime type > GPU).

Flow: install deps -> mount Drive -> run 10 hyperparameter experiments + 1 MLP-vs-CNN comparison -> download results -> (after picking the best config) train one longer 'champion' model for the play.py demo.

In [ ]:
!pip install -q "stable-baselines3[extra]" gymnasium ale-py opencv-python autorom[accept-rom-license]
!AutoROM --accept-license -y

In [ ]:
# Mount Google Drive so models/experiment CSVs survive if the Colab session disconnects
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/dqn_boxing_wagner'
os.makedirs(f'{SAVE_DIR}/models', exist_ok=True)
os.makedirs(f'{SAVE_DIR}/experiments', exist_ok=True)
os.chdir(SAVE_DIR)
os.makedirs('models', exist_ok=True)
os.makedirs('experiments', exist_ok=True)
print('Working dir:', os.getcwd())

In [ ]:
import gymnasium as gym
import ale_py
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.evaluation import evaluate_policy
import csv

gym.register_envs(ale_py)
ENV_ID = "ALE/Boxing-v5"

def create_env(n_envs=1, seed=42):
    env = make_atari_env(ENV_ID, n_envs=n_envs, seed=seed, env_kwargs={"frameskip": 1})
    return VecFrameStack(env, n_stack=4)

def train_agent(policy="CnnPolicy", learning_rate=1e-4, gamma=0.99, batch_size=32,
                 exploration_initial_eps=1.0, exploration_final_eps=0.05, exploration_fraction=0.1,
                 total_timesteps=150_000, run_name="baseline"):
    env = create_env()
    model = DQN(policy=policy, env=env, learning_rate=learning_rate, gamma=gamma,
                batch_size=batch_size, exploration_initial_eps=exploration_initial_eps,
                exploration_final_eps=exploration_final_eps, exploration_fraction=exploration_fraction,
                buffer_size=100_000, learning_starts=10_000, target_update_interval=1_000,
                train_freq=4, verbose=1, tensorboard_log="logs/tensorboard/")
    model.learn(total_timesteps=total_timesteps, tb_log_name=run_name)
    model.save(f"models/dqn_model_{run_name}.zip")
    env.close()
    return model

def evaluate_agent(model, n_episodes=5):
    eval_env = create_env(seed=100)
    mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=n_episodes, deterministic=True)
    eval_env.close()
    return mean_reward, std_reward

def already_run(member, run_name):
    """Check whether run_name has already been logged (lets a crashed sweep resume safely)."""
    filepath = f"experiments/{member}_experiments.csv"
    if not os.path.isfile(filepath):
        return False
    with open(filepath, newline="") as f:
        return run_name in {row["run_name"] for row in csv.DictReader(f)}

def log_experiment(member, run_name, params, mean_reward, std_reward):
    filepath = f"experiments/{member}_experiments.csv"
    file_exists = os.path.isfile(filepath)
    with open(filepath, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["run_name", "policy", "lr", "gamma", "batch_size",
                              "eps_start", "eps_end", "eps_fraction", "mean_reward", "std_reward"])
        writer.writerow([run_name, params.get("policy", "CnnPolicy"), params["learning_rate"], params["gamma"],
                          params["batch_size"], params["exploration_initial_eps"],
                          params["exploration_final_eps"], params["exploration_fraction"],
                          round(mean_reward, 2), round(std_reward, 2)])

print('Functions ready.')

In [ ]:
MEMBER = "wagner"
TIMESTEPS_PER_RUN = 150_000  # lower to 100_000 if you're short on time

EXPERIMENTS = [
    dict(run_name="exp01_baseline", policy="CnnPolicy", params=dict(
        learning_rate=1e-4, gamma=0.99, batch_size=32,
        exploration_initial_eps=1.0, exploration_final_eps=0.05, exploration_fraction=0.1)),
    dict(run_name="exp02_lr_mid_high", policy="CnnPolicy", params=dict(
        learning_rate=5e-4, gamma=0.99, batch_size=32,
        exploration_initial_eps=1.0, exploration_final_eps=0.05, exploration_fraction=0.1)),
    dict(run_name="exp03_lr_very_low", policy="CnnPolicy", params=dict(
        learning_rate=5e-6, gamma=0.99, batch_size=32,
        exploration_initial_eps=1.0, exploration_final_eps=0.05, exploration_fraction=0.1)),
    dict(run_name="exp04_gamma_high", policy="CnnPolicy", params=dict(
        learning_rate=1e-4, gamma=0.999, batch_size=32,
        exploration_initial_eps=1.0, exploration_final_eps=0.05, exploration_fraction=0.1)),
    dict(run_name="exp05_gamma_mid", policy="CnnPolicy", params=dict(
        learning_rate=1e-4, gamma=0.95, batch_size=32,
        exploration_initial_eps=1.0, exploration_final_eps=0.05, exploration_fraction=0.1)),
    dict(run_name="exp06_batch_very_high", policy="CnnPolicy", params=dict(
        learning_rate=1e-4, gamma=0.99, batch_size=128,
        exploration_initial_eps=1.0, exploration_final_eps=0.05, exploration_fraction=0.1)),
    dict(run_name="exp07_batch_very_low", policy="CnnPolicy", params=dict(
        learning_rate=1e-4, gamma=0.99, batch_size=8,
        exploration_initial_eps=1.0, exploration_final_eps=0.05, exploration_fraction=0.1)),
    dict(run_name="exp08_eps_start_low", policy="CnnPolicy", params=dict(
        learning_rate=1e-4, gamma=0.99, batch_size=32,
        exploration_initial_eps=0.5, exploration_final_eps=0.05, exploration_fraction=0.1)),
    dict(run_name="exp09_eps_end_very_low", policy="CnnPolicy", params=dict(
        learning_rate=1e-4, gamma=0.99, batch_size=32,
        exploration_initial_eps=1.0, exploration_final_eps=0.01, exploration_fraction=0.2)),
    dict(run_name="exp10_tuned_combo", policy="CnnPolicy", params=dict(
        # EDIT before running: combine whichever changes helped most in exp01-09
        learning_rate=5e-4, gamma=0.999, batch_size=128,
        exploration_initial_eps=1.0, exploration_final_eps=0.01, exploration_fraction=0.2)),
]

MLP_COMPARISON = dict(run_name="exp_mlp_vs_cnn", policy="MlpPolicy", params=dict(
    learning_rate=1e-4, gamma=0.99, batch_size=32,
    exploration_initial_eps=1.0, exploration_final_eps=0.05, exploration_fraction=0.1))

print(f"{len(EXPERIMENTS) + 1} runs queued, {TIMESTEPS_PER_RUN} timesteps each.")

In [ ]:
# Run all 10 hyperparameter experiments + the MLP comparison.
# This is the long-running cell -- expect roughly 20-40 min per 150k-step run on a T4 GPU.
# Safe to re-run after a crash/disconnect: already-logged run_names are skipped automatically.
for exp in EXPERIMENTS + [MLP_COMPARISON]:
    if already_run(MEMBER, exp["run_name"]):
        print(f"\n=== Skipping {exp['run_name']} (already logged) ===")
        continue
    print(f"\n=== Running {exp['run_name']} ({exp['policy']}) ===")
    model = train_agent(policy=exp["policy"], total_timesteps=TIMESTEPS_PER_RUN,
                         run_name=exp["run_name"], **exp["params"])
    mean_reward, std_reward = evaluate_agent(model)
    log_params = dict(exp["params"]); log_params["policy"] = exp["policy"]
    log_experiment(MEMBER, exp["run_name"], log_params, mean_reward, std_reward)
    print(f"{exp['run_name']}: mean reward = {mean_reward:.2f} +/- {std_reward:.2f}")

In [ ]:
# Inspect results so far
import pandas as pd
df = pd.read_csv(f"experiments/{MEMBER}_experiments.csv")
df.sort_values("mean_reward", ascending=False)

## Champion run
After reviewing the table above, pick the single best-performing hyperparameter
combo and retrain it for much longer (e.g. 800k-1,000,000 steps) so the final
model actually plays well for the play.py demo/recording. Edit BEST_PARAMS below
to match your winning row before running.

In [ ]:
BEST_PARAMS = dict(
    learning_rate=1e-4,   # <-- replace with your best row's values
    gamma=0.99,
    batch_size=32,
    exploration_initial_eps=1.0,
    exploration_final_eps=0.05,
    exploration_fraction=0.1,
)

champion_model = train_agent(policy="CnnPolicy", total_timesteps=800_000,
                              run_name="wagner_best", **BEST_PARAMS)
mean_reward, std_reward = evaluate_agent(champion_model, n_episodes=10)
print(f"champion: mean reward = {mean_reward:.2f} +/- {std_reward:.2f}")
print("Saved to models/dqn_model_wagner_best.zip -- download this and put it in")
print("your local repo's models/ folder, then run play.py locally to watch it play.")

In [ ]:
# Everything is already saved under /content/drive/MyDrive/dqn_boxing_wagner/
# (models/ and experiments/) since we cd'ed there earlier -- just grab the files
# from Google Drive directly, or download explicitly here:
from google.colab import files
files.download("models/dqn_model_wagner_best.zip")
files.download(f"experiments/{MEMBER}_experiments.csv")